# 06: scikit-learn Introduction - Professional ML Tools

## From Scratch to Production-Ready

We've built everything from scratch - which is great for understanding! But in practice, we use battle-tested libraries.

**scikit-learn** (sklearn) is the most popular ML library in Python:
- Consistent API across all algorithms
- Optimized implementations
- Built-in preprocessing and evaluation tools

### The Web Dev Analogy

sklearn is like using **React/Vue/Angular** instead of vanilla JavaScript:
- Same concepts (components/models)
- Much more efficient (battle-tested implementations)
- Consistent patterns (fit/predict/transform)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The sklearn imports we'll use
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("scikit-learn ready! 🔧")

## 1. The sklearn API Pattern

Almost everything in sklearn follows the same pattern:

```python
# Create model
model = SomeModel(parameters)

# Train on data
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# For transformers (preprocessors)
X_transformed = transformer.fit_transform(X_train)
```

This consistency is powerful - once you learn one model, you know them all!

## 2. Our Dataset (Again)

In [ ]:
# Same movie review dataset
reviews = [
    # Positive reviews
    ("This movie was absolutely amazing! I loved every minute.", 1),
    ("Fantastic film with great acting. Highly recommended!", 1),
    ("A wonderful story that moved me to tears. Beautiful!", 1),
    ("Best movie I've seen this year. Outstanding performance.", 1),
    ("Incredible cinematography and a touching story. Loved it!", 1),
    ("The acting was superb and the plot was engaging.", 1),
    ("A masterpiece! This film exceeded all expectations.", 1),
    ("Heartwarming and funny. A perfect feel-good movie.", 1),
    ("Brilliant direction and stellar performances throughout.", 1),
    ("This movie made me laugh and cry. Absolutely wonderful!", 1),
    ("An excellent film that keeps you engaged from start to finish.", 1),
    ("The best movie of the decade. A true cinematic achievement.", 1),
    
    # Negative reviews
    ("Terrible movie. Complete waste of time and money.", 0),
    ("Boring and predictable. I fell asleep halfway through.", 0),
    ("The worst film I've ever seen. Awful acting.", 0),
    ("Disappointing and dull. Not worth watching.", 0),
    ("A disaster of a movie. Poor writing and bad direction.", 0),
    ("I hated this film. It was painfully slow and boring.", 0),
    ("Waste of money. The plot made no sense at all.", 0),
    ("Horrible acting and a ridiculous storyline. Avoid!", 0),
    ("This movie was a complete disappointment. So bad.", 0),
    ("Unwatchable garbage. I want my two hours back.", 0),
    ("A terrible mess from start to finish. Just awful.", 0),
    ("The acting was wooden and the story was nonsensical.", 0),
]

texts = [r[0] for r in reviews]
labels = np.array([r[1] for r in reviews])

# Split data - sklearn makes this easy!
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

print(f"Training: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")

## 3. CountVectorizer: Bag of Words Made Easy

sklearn's `CountVectorizer` does what we built manually, but better:

In [ ]:
# Create vectorizer
vectorizer = CountVectorizer(
    lowercase=True,           # Convert to lowercase
    stop_words='english',     # Remove common words like 'the', 'is'
    min_df=2,                 # Ignore words that appear < 2 times
    max_df=0.9,               # Ignore words that appear in > 90% of docs
)

# Fit on training data, transform both
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)  # Note: transform only, not fit_transform!

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Training matrix shape: {X_train_vec.shape}")
print(f"Test matrix shape: {X_test_vec.shape}")

# Show some vocabulary
print(f"\nSample vocabulary words: {list(vectorizer.vocabulary_.keys())[:15]}")

## 4. TF-IDF: Smarter Word Weighting

**TF-IDF** (Term Frequency - Inverse Document Frequency) gives more weight to distinctive words:

- **TF**: How often does the word appear in this document?
- **IDF**: How rare is this word across all documents?

Words like "the" get low scores (common everywhere), while distinctive words get high scores.

In [ ]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=2,
    max_df=0.9,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

# Compare CountVectorizer vs TF-IDF for one document
print("\nComparison for first training document:")
print(f"CountVectorizer - max value: {X_train_vec[0].max():.2f}")
print(f"TF-IDF - max value: {X_train_tfidf[0].max():.4f}")
print("\n→ TF-IDF values are normalized and weighted!")

## 5. Logistic Regression with sklearn

In [ ]:
# Create and train model - it's this simple!
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.2%}")

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 6. Trying Different Models

The sklearn API makes it easy to try different algorithms:

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Dictionary of models to try
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(max_iter=1000),
}

print("Comparing different models:")
print("=" * 40)

results = {}
for name, model in models.items():
    # Train
    model.fit(X_train_tfidf, y_train)
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:25} Accuracy: {acc:.2%}")

# Visualize
plt.figure(figsize=(10, 4))
plt.barh(list(results.keys()), list(results.values()), color='steelblue')
plt.xlabel('Accuracy')
plt.title('Model Comparison')
plt.xlim(0, 1)
for i, (name, acc) in enumerate(results.items()):
    plt.text(acc + 0.02, i, f'{acc:.1%}', va='center')
plt.tight_layout()
plt.show()

## 7. Pipelines: Combining Steps

Pipelines chain preprocessing and modeling together - cleaner and prevents data leakage!

In [ ]:
# Create a pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', min_df=2)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Now we can work with raw text directly!
pipeline.fit(X_train, y_train)

# Predict on raw text
y_pred = pipeline.predict(X_test)
print(f"Pipeline accuracy: {accuracy_score(y_test, y_pred):.2%}")

# Predict on new reviews directly!
new_reviews = [
    "This was an amazing experience!",
    "Terrible waste of time."
]

predictions = pipeline.predict(new_reviews)
probas = pipeline.predict_proba(new_reviews)

print("\nPredictions on new reviews:")
for review, pred, proba in zip(new_reviews, predictions, probas):
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = max(proba)
    print(f"  '{review}' → {sentiment} ({confidence:.1%} confidence)")

## 8. Feature Importance (Interpretability)

In [ ]:
# Get the components from the pipeline
vectorizer = pipeline.named_steps['vectorizer']
classifier = pipeline.named_steps['classifier']

# Get feature names and coefficients
feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

# Sort by coefficient
sorted_idx = np.argsort(coefficients)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Top negative words
n_top = 10
neg_idx = sorted_idx[:n_top]
axes[0].barh(range(n_top), coefficients[neg_idx], color='red', alpha=0.7)
axes[0].set_yticks(range(n_top))
axes[0].set_yticklabels(feature_names[neg_idx])
axes[0].set_xlabel('Coefficient')
axes[0].set_title('Top Negative Indicators')

# Top positive words
pos_idx = sorted_idx[-n_top:][::-1]
axes[1].barh(range(n_top), coefficients[pos_idx], color='green', alpha=0.7)
axes[1].set_yticks(range(n_top))
axes[1].set_yticklabels(feature_names[pos_idx])
axes[1].set_xlabel('Coefficient')
axes[1].set_title('Top Positive Indicators')

plt.tight_layout()
plt.show()

## 9. Cross-Validation: More Reliable Evaluation

A single train/test split can be misleading. **Cross-validation** gives more reliable estimates:

In [ ]:
from sklearn.model_selection import cross_val_score

# Create pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', min_df=1)),  # min_df=1 for small dataset
    ('classifier', LogisticRegression(max_iter=1000))
])

# 5-fold cross-validation
scores = cross_val_score(pipeline, texts, labels, cv=5, scoring='accuracy')

print("5-Fold Cross-Validation Results:")
print(f"Scores: {scores}")
print(f"Mean: {scores.mean():.2%} (+/- {scores.std()*2:.2%})")

## 10. Saving and Loading Models

For production, use `joblib` which is more efficient for sklearn models:

In [ ]:
import joblib
import os

# Train final model
final_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', min_df=1)),
    ('classifier', LogisticRegression(max_iter=1000))
])
final_pipeline.fit(texts, labels)  # Train on all data

# Save model using joblib (recommended for sklearn)
model_path = '../../data/sentiment_model.joblib'
os.makedirs(os.path.dirname(model_path), exist_ok=True)

joblib.dump(final_pipeline, model_path)
print(f"Model saved to {model_path}")

# Load model
loaded_model = joblib.load(model_path)

# Verify it works
test_text = "This is an absolutely wonderful movie!"
prediction = loaded_model.predict([test_text])[0]
print(f"\nLoaded model prediction: '{test_text}' → {'Positive' if prediction else 'Negative'}")

## 📝 Check Your Understanding

1. What's the difference between `fit()`, `transform()`, and `fit_transform()`?
2. Why should we only `fit` on training data, not test data?
3. What's the advantage of TF-IDF over simple word counts?
4. Why use pipelines instead of separate steps?
5. What does cross-validation give us that a single split doesn't?

## 🎯 Summary

You learned to use sklearn for:
- **Vectorization**: CountVectorizer, TfidfVectorizer
- **Classification**: LogisticRegression, MultinomialNB, LinearSVC
- **Evaluation**: accuracy_score, classification_report, cross_val_score
- **Pipelines**: Combining preprocessing + modeling
- **Persistence**: Saving and loading models with joblib

The consistent `fit/predict/transform` API makes it easy to try different approaches!

**Next up**: Neural Networks from scratch - where the magic really begins! →